In [0]:
%run ../../Configurations/Init_Scripts

Fraud Classification

In [0]:
input_path = '/mnt/raw/MasterData/FraudClassification'
output_path = '/mnt/silver/DIMFraudClassification'
df_FraudClassification = spark.read.csv(input_path,header=True)
PromotionUUID= spark.sql("select PromotionUUID from promotion.dim_promotion where PromotionName = '4P Beta' " ).first()[0]

df_FraudClassification = (df_FraudClassification
                            .withColumn('FraudClassificationUUID',expr('uuid()'))
                            .withColumn("PromotionUUID",lit(PromotionUUID))
                            .withColumn('CreatedBy',lit('ADB_DIM_FraudClassification'))
                            .withColumn('CreatedDate',current_timestamp())
                            .withColumn('UpdatedBy',lit('ADB_DIM_FraudClassification'))
                            .withColumn('UpdatedDate',current_timestamp())
                            .select('FraudClassificationUUID','PromotionUUID','RuleName','RuleDetails','DisplayFlag','ThresholdLimit',
                                    'CreatedDate','CreatedBy','UpdatedDate','UpdatedBy'))


In [0]:
dl_Target = DeltaTable.forPath(spark, output_path)  
(dl_Target.alias("tgt")
  .merge(df_FraudClassification.alias("src"), "src.RuleName = tgt.RuleName and src.PromotionUUID = tgt.PromotionUUID ")
  .whenMatchedUpdate(
    set = {
        "tgt.RuleDetails": "src.RuleDetails",
        "tgt.DisplayFlag": "src.DisplayFlag",
        "tgt.ThresholdLimit": "src.ThresholdLimit",
        "tgt.UpdatedDate": "src.UpdatedDate",
        "tgt.UpdatedBy": "src.UpdatedBy"
           }
  )
  .whenNotMatchedInsertAll()
  .execute()
)